In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [4]:
df = pd.read_csv('/content/qoute_dataset.csv')


In [5]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [6]:
quotes = df['quote']

In [7]:
quotes = quotes.str.lower()

In [8]:
import string
translater = str.maketrans("", "",string.punctuation)

In [9]:
quotes = quotes.apply(lambda x: x.translate(translater))


In [10]:
quotes

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...
...,...
3033,the past beats inside me like a second heart
3034,damn claire warn a guy before you do a facepla...
3035,can you be a girl for a few secondsim always a...
3036,thats what fiction is for its for getting at t...


In [11]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [12]:
vocab_size = 10000
tokinizer = Tokenizer(num_words=vocab_size)
tokinizer.fit_on_texts(quotes)

In [13]:
word_index = tokinizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [14]:
sequence = tokinizer.texts_to_sequences(quotes)

In [15]:
for i in range(3):
  print(quotes[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [16]:
for i in range(3):
  print(sequence[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


In [17]:
X = []
y = []

for seq in sequence:
  for i in range(1,len(seq)):
    input_seq = seq[:i]
    output_seq = seq[i]
    X.append(input_seq)
    y.append(output_seq)



In [18]:
max_len = max(len(x) for x in X)
print(max_len)

745


In [19]:
from keras.utils import pad_sequences
X_pad = pad_sequences(X,maxlen=max_len,padding='pre')

In [20]:
y = np.array(y)

In [21]:
from tensorflow.keras.utils import to_categorical
y_one = to_categorical(y,num_classes=vocab_size)

In [22]:
y_one

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [23]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,LSTM,Dense

In [24]:
embedding_dim = 50
rnn_units = 128

In [25]:
rnn_model = Sequential()

rnn_model.add(Embedding(input_dim=vocab_size,output_dim=embedding_dim,input_length=max_len))
rnn_model.add(SimpleRNN(units=rnn_units))
rnn_model.add(Dense(units=vocab_size,activation='softmax'))



/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [26]:
rnn_model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

In [27]:
rnn_model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [28]:
lstm_model = Sequential()
lstm_model.add(Embedding(input_dim=vocab_size,output_dim=embedding_dim,input_length=max_len))
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size,activation='softmax'))

In [29]:
lstm_model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

In [30]:
epoxhs = 15
bach_size = 128

In [31]:
history = rnn_model.fit(X_pad,y_one,epochs=epoxhs,batch_size=bach_size,validation_split=0.1)

Epoch 1/15
600/600 ━━━━━━━━━━━━━━━━━━━━ 47s 71ms/step - accuracy: 0.0468 - loss: 6.6932 - val_accuracy: 0.0627 - val_loss: 6.4926
Epoch 2/15
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 64ms/step - accuracy: 0.0810 - loss: 6.0672 - val_accuracy: 0.0932 - val_loss: 6.2980
Epoch 3/15
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 63ms/step - accuracy: 0.1035 - loss: 5.7108 - val_accuracy: 0.0985 - val_loss: 6.2988
Epoch 4/15
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 63ms/step - accuracy: 0.1183 - loss: 5.4271 - val_accuracy: 0.1072 - val_loss: 6.3236
Epoch 5/15
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 63ms/step - accuracy: 0.1309 - loss: 5.1675 - val_accuracy: 0.1076 - val_loss: 6.3871
Epoch 6/15
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 63ms/step - accuracy: 0.1457 - loss: 4.9258 - val_accuracy: 0.1089 - val_loss: 6.4245
Epoch 7/15
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 63ms/step - accuracy: 0.1633 - loss: 4.6974 - val_accuracy: 0.1069 - val_loss: 6.5125
Epoch 8/15
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 63ms/step - accuracy: 0.1817 - loss: 4.4863 - 